# Step 5: Feature Engineering
Build the 11 candidate features (Step 6 keeps 8). Every employee / department / category statistic uses **earlier days only** (mentor feedback MF1: no leakage), and new employees follow a written fallback rule (cold start, N = 5).

**Input:** `data/processed/requests.csv`
**Output:** `data/processed/features.csv` ((209,012 rows × 18 columns, after removing 13 bulk buyers)

| Part | Cells | What |
|---|---|---|
| A | 5.1-5.5 | Statistics from earlier requests only |
| B | 5.6-5.9 | Amount z-scores + the rule for new employees |
| C | 5.10-5.14 | Category share, same-day counts, weekend, round amounts, save |

In [1]:
# 5.1 Load the requests and sort them by date (oldest first): the statistics below need time order.
# log_amount = log(1 + amount). Amounts are very skewed; the log makes them comparable.
# log_amount_sq (the square) is needed to get the spread (std) from running sums in 5.2.
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/requests.csv",
                 parse_dates=["expense_date", "created_at"])
df = df.sort_values(["expense_date", "expense_id"]).reset_index(drop=True)

df["log_amount"] = np.log1p(df["amount_sar"])
df["log_amount_sq"] = df["log_amount"] ** 2
print("Loaded:", df.shape)

Loaded: (247838, 14)


In [2]:
# 5.1b Remove the bulk buyers (decided in Step 6)
# 13 cardholders usually send 5+ requests a day (bulk travel / equipment buyers on shared cards,
# not personal expenses). They're removed BEFORE the statistics, so the department and category
# averages are clean too. The analysis is in notebook 05 (cells 6.2-6.4).
# Keep BUSY_IDS: the same 13 are left out of the demo seed files.
BUSY_FROM = 5

per_day = df.groupby(["requester_id", "expense_date"]).size()
usual_day = per_day.groupby("requester_id").median()
BUSY_IDS = sorted(usual_day[usual_day >= BUSY_FROM].index)

print("Removing", len(BUSY_IDS), "busy employees:", BUSY_IDS)
print("Requests removed:", df["requester_id"].isin(BUSY_IDS).sum())

df = df[~df["requester_id"].isin(BUSY_IDS)].reset_index(drop=True)
print("Left:", df.shape)

Removing 13 busy employees: [1194, 1237, 1752, 1759, 2712, 4097, 4438, 4481, 4851, 5060, 5112, 5438, 5571]
Requests removed: 38826
Left: (209012, 14)


## Part A: Statistics from earlier requests only
For each request: how many requests the same employee / department / category had on **earlier days**, and the average and spread of their log amounts. Requests on the same day don't see each other, and no future request is ever used.

In [3]:
# 5.2 prior_stats: for one kind of group (e.g. per employee), stats from EARLIER DAYS only.
# 1) day = one row per group per day: n (count), s (sum of log amounts), s2 (sum of squares).
# 2) cumsum() = running total up to and including that day; subtracting the day itself
#    leaves only the earlier days.
# 3) mean = s / n, variance = (s2 - n * mean^2) / (n - 1). .where(n > 0) gives NaN instead of
#    dividing by zero when there's no history yet. clip(lower=0) removes tiny negative rounding errors.
# 4) Merge the day-level stats back onto every request of that group and day.
def prior_stats(df, keys, name):
    day = (df.groupby(keys + ["expense_date"], as_index=False)
             .agg(n=("log_amount", "size"),
                  s=("log_amount", "sum"),
                  s2=("log_amount_sq", "sum")))
    day = day.sort_values(keys + ["expense_date"])
    g = day.groupby(keys)

    n_prev  = g["n"].cumsum()  - day["n"]
    s_prev  = g["s"].cumsum()  - day["s"]
    s2_prev = g["s2"].cumsum() - day["s2"]

    mean = s_prev / n_prev.where(n_prev > 0)
    var  = (s2_prev - n_prev * mean ** 2) / (n_prev - 1).where(n_prev > 1)

    day[f"{name}_n"]    = n_prev
    day[f"{name}_mean"] = mean
    day[f"{name}_std"]  = np.sqrt(var.clip(lower=0))

    cols = keys + ["expense_date", f"{name}_n", f"{name}_mean", f"{name}_std"]
    return df.merge(day[cols], on=keys + ["expense_date"], how="left")

In [4]:
# 5.3 Run prior_stats for the 5 groups (each adds <name>_n, <name>_mean, <name>_std):
#   emp = employee, empcat = employee in this category, dept = department,
#   deptcat = department in this category, cat = the whole category.
# Then show one employee's first requests, to see the history grow (n goes 0, 1, 2, ...).
GROUPS = {
    "emp":     ["requester_id"],
    "empcat":  ["requester_id", "category"],
    "dept":    ["department_id"],
    "deptcat": ["department_id", "category"],
    "cat":     ["category"],
}
for name, keys in GROUPS.items():
    df = prior_stats(df, keys, name)
print(df.shape)

example_id = df["requester_id"].iloc[1000]
print("Employee", example_id)
ex = df[df["requester_id"] == example_id].head(6)
print(ex[["expense_date", "category", "amount_sar", "log_amount", "emp_n", "emp_mean", "emp_std"]]
        .round(3).to_string(index=False))

(209012, 29)
Employee 2928
expense_date                category  amount_sar  log_amount  emp_n  emp_mean  emp_std
  2025-06-27 Equipment & Maintenance     1090.09       6.995      0       NaN      NaN
  2025-06-27 Equipment & Maintenance     2246.66       7.718      0       NaN      NaN
  2025-08-13          Transportation      361.72       5.894      2     7.356    0.511


C:\Users\Zala\AppData\Local\Temp\ipykernel_32096\2077806299.py:20: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  .round(3).to_string(index=False))


In [5]:
# 5.4 Brute-force check: for 300 random requests, recompute every stat the slow, obvious way
# (keep the same group's rows from earlier days, then .mean() / .std()) and compare with prior_stats.
# 0 mismatches = the fast version is correct and never looks at the same day or the future.
rng = np.random.default_rng(0)
sample = df.iloc[rng.choice(len(df), size=300, replace=False)]

mismatches = 0
for _, row in sample.iterrows():
    for name, keys in GROUPS.items():
        same_group = np.ones(len(df), dtype=bool)
        for k in keys:
            same_group &= (df[k] == row[k]).to_numpy()
        earlier = df.loc[same_group & (df["expense_date"] < row["expense_date"]).to_numpy(), "log_amount"]
        if len(earlier) != row[f"{name}_n"]:
            mismatches += 1
        elif len(earlier) > 0 and not np.isclose(earlier.mean(), row[f"{name}_mean"]):
            mismatches += 1
        elif len(earlier) > 1 and not np.isclose(earlier.std(), row[f"{name}_std"], atol=1e-6):
            mismatches += 1
print("Checked", len(sample) * len(GROUPS), "values | mismatches:", mismatches)

Checked 1500 values | mismatches: 0


In [6]:
# 5.5 How much history the requests have: the share with no history at all, and with fewer than
# 5 earlier requests (that's where the new-employee rule in Part B is needed).
for name in GROUPS:
    print(f"{name:8} no history: {(df[f'{name}_n'] == 0).mean():6.1%} | < 5 earlier: {(df[f'{name}_n'] < 5).mean():6.1%}")

emp      no history:   2.9% | < 5 earlier:  10.7%
empcat   no history:   7.4% | < 5 earlier:  22.9%
dept     no history:   0.0% | < 5 earlier:   0.1%
deptcat  no history:   0.1% | < 5 earlier:   0.5%
cat      no history:   0.0% | < 5 earlier:   0.1%


## Part B: Amount features + the rule for new employees
z-score = how far this amount is from the group's normal, counted in spreads (0 = normal, +3 = much higher than usual). A level is only used when it has at least `MIN_HISTORY` earlier requests; otherwise the next, broader level is used.

In [7]:
# 5.6 Settings (in Step 9 they move to src/features.py, shared by the notebooks and the ML service):
#   MIN_HISTORY = trust an average only if it has >= 5 earlier requests (the "N" of the cold-start rule).
#   MIN_STD = minimum spread. Someone who always paid the same amount has spread 0, and dividing
#             by 0 gives infinity (employee 2099: 112.46 every time).
#   Z_CAP = anything beyond ±10 is just "extremely unusual"; capping stops extremes from distorting the model.
# Changing any of these means rebuilding the features AND retraining the model.
# zscore(level) = (this log amount - the level's earlier average) / the level's earlier spread.
MIN_HISTORY = 5
MIN_STD = 0.1
Z_CAP = 10

def zscore(level):
    std = df[f"{level}_std"].clip(lower=MIN_STD)
    z = (df["log_amount"] - df[f"{level}_mean"]) / std
    return z.clip(-Z_CAP, Z_CAP)

In [8]:
# 5.7 The rule for new employees (cold start): for each request, use the FIRST level in the list
# that has >= MIN_HISTORY earlier requests. np.select works like if / elif / elif for every row at once.
# `used` records which level was used. No level with enough history (only the first days of the
# data) → z = 0, treated as normal.
def cascade(levels):
    conditions = [df[f"{lv}_n"] >= MIN_HISTORY for lv in levels]
    values = [zscore(lv) for lv in levels]
    z = np.select(conditions, values, default=np.nan)
    used = np.select(conditions, levels, default="none")
    return pd.Series(z, index=df.index).fillna(0.0), pd.Series(used, index=df.index)

In [9]:
# 5.8 Build the 3 amount features, each with its own fallback order (most specific first).
# baseline_level = which level amount_vs_employee_category used. It's saved for the reasons later,
# e.g. "compared with the department because this employee is new". `_` = "don't need this value".
df["amount_vs_employee_category"], df["baseline_level"] = cascade(["empcat", "emp", "deptcat", "cat"])
df["amount_vs_employee"], _ = cascade(["emp", "dept", "cat"])
df["amount_vs_department"], _ = cascade(["dept", "cat"])

print((df["baseline_level"].value_counts(normalize=True) * 100).round(1))

baseline_level
empcat     77.1
emp        12.3
deptcat    10.2
cat         0.4
none        0.1
Name: proportion, dtype: float64


In [10]:
# 5.9 Checks: mean ~0 and std ~1 (a sign the z-scores are right), how many values hit the cap,
# and employee 3170's history: on 8 Dec, 744.90 Transportation = 5.87 for the employee (very unusual)
# but 0.69 for the department (normal). That's why we keep both.
AMOUNT_FEATURES = ["amount_vs_employee_category", "amount_vs_employee", "amount_vs_department"]
print(df[AMOUNT_FEATURES].describe().round(2))
print("At the cap (|z| = 10):", (df[AMOUNT_FEATURES].abs() == Z_CAP).sum().to_dict())

ex = df[df["requester_id"] == 3170][["expense_date", "category", "amount_sar",
                                     "baseline_level"] + AMOUNT_FEATURES]
print(ex.round(2).to_string(index=False))

       amount_vs_employee_category  amount_vs_employee  amount_vs_department
count                    209012.00           209012.00             209012.00
mean                         -0.00               -0.00                  0.01
std                           1.20                1.11                  1.00
min                         -10.00              -10.00                 -4.27
25%                          -0.75               -0.73                 -0.71
50%                          -0.01               -0.02                 -0.03
75%                           0.74                0.71                  0.68
max                          10.00               10.00                  4.93
At the cap (|z| = 10): {'amount_vs_employee_category': 173, 'amount_vs_employee': 65, 'amount_vs_department': 0}
expense_date                category  amount_sar baseline_level  amount_vs_employee_category  amount_vs_employee  amount_vs_department
  2025-08-02                    Food       68.21        dep

C:\Users\Zala\AppData\Local\Temp\ipykernel_32096\2444041442.py:10: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(ex.round(2).to_string(index=False))


## Part C: Category share, same-day counts, weekend, round amounts
The other signals from the contract: an unusual category, splits, duplicates, many requests in one day, weekends and round amounts. Then save `features.csv`.

In [11]:
# 5.10 category_share_for_employee = of this employee's EARLIER requests, the share in this category
# (1.0 = always uses it, 0 = first time ever). Fewer than 5 earlier requests → the department's share
# instead; no history at all → 1.0 (not unusual). It reuses the _n counts from Part A, so it's earlier-only too.
emp_share  = df["empcat_n"] / df["emp_n"]
dept_share = df["deptcat_n"] / df["dept_n"]

df["category_share_for_employee"] = np.select(
    [df["emp_n"] >= MIN_HISTORY, df["dept_n"] >= MIN_HISTORY],
    [emp_share, dept_share],
    default=1.0)

print(df["category_share_for_employee"].describe().round(3))
print("Share = 0 (first time in this category):", (df["category_share_for_employee"] == 0).sum())

ex = df[df["requester_id"] == 3170][["expense_date", "category", "emp_n", "empcat_n",
                                     "category_share_for_employee"]].copy()
ex["category_share_for_employee"] = ex["category_share_for_employee"].round(2)
print(ex.to_string(index=False))

count    209012.000
mean          0.566
std           0.320
min           0.000
25%           0.273
50%           0.589
75%           0.870
max           1.000
Name: category_share_for_employee, dtype: float64
Share = 0 (first time in this category): 5066
expense_date                category  emp_n  empcat_n  category_share_for_employee
  2025-08-02                    Food      0         0                         0.06
  2025-08-04                    Food      1         1                         0.07
  2025-08-06                    Food      2         2                         0.08
  2025-08-08                    Food      3         3                         0.08
  2025-08-10                    Food      4         4                         0.08
  2025-10-06                    Food      5         5                         1.00
  2025-10-08                    Food      6         6                         1.00
  2025-10-10                    Food      7         7                         1.

In [12]:
# 5.11 Same-day counts (each count includes the request itself, so 1 = only this one):
#   daily_request_count     = same employee, same day   → many requests in one day
#   same_category_day_count = + same category          → splitting one purchase to stay under the limit
#   duplicate_count         = + same amount            → the same request submitted twice
# transform("size") counts the rows in each group and writes the count on every row of that group.
# These use the WHOLE day on purpose: in the app, the backend re-scores the other Pending requests of
# that day (contract C3), so in the end each request is scored with the full day too.
day = ["requester_id", "expense_date"]

df["daily_request_count"]     = df.groupby(day)["expense_id"].transform("size")
df["same_category_day_count"] = df.groupby(day + ["category"])["expense_id"].transform("size")
df["duplicate_count"]         = df.groupby(day + ["category", "amount_sar"])["expense_id"].transform("size")

# other_category_day_count = requests the same day in OTHER categories (decided in Step 6, cell 6.10).
# daily_request_count was 0.97 correlated with same_category_day_count (on most days people use
# one category), so we keep only the part that's new. daily_request_count stays in the file, but
# Step 6 doesn't select it.
df["other_category_day_count"] = df["daily_request_count"] - df["same_category_day_count"]

for c in ["daily_request_count", "same_category_day_count", "duplicate_count"]:
    print(c, "| more than 1:", (df[c] > 1).sum(), "| max:", df[c].max())
print("other_category_day_count | more than 0:", (df["other_category_day_count"] > 0).sum(),
      "| max:", df["other_category_day_count"].max())

daily_request_count | more than 1: 115264 | max: 80
same_category_day_count | more than 1: 95563 | max: 80
duplicate_count | more than 1: 15852 | max: 39
other_category_day_count | more than 0: 39115 | max: 57


In [13]:
# Check a real split (employee 1157: 3 Food requests on one day, each under the 2,000 limit,
# 2,021.25 together) and a duplicate (employee 1001: the same 1,322.25 Travel twice on one day).
cols = ["expense_id", "category", "amount_sar", "same_category_day_count", "duplicate_count", "daily_request_count"]
print(df[(df["requester_id"] == 1157) & (df["expense_date"] == "2026-01-14")][cols].to_string(index=False))
print(df[(df["requester_id"] == 1001) & (df["expense_date"] == "2025-08-20")][cols].to_string(index=False))

 expense_id category  amount_sar  same_category_day_count  duplicate_count  daily_request_count
     231604     Food      544.61                        3                1                    3
     231605     Food      688.84                        3                1                    3
     231606     Food      787.80                        3                1                    3
 expense_id category  amount_sar  same_category_day_count  duplicate_count  daily_request_count
     137287   Travel     1322.25                        2                2                    2
     137288   Travel     1322.25                        2                2                    2


In [14]:
# 5.12 is_weekend = Friday (4) or Saturday (5); in pandas Monday = 0.
# is_round_amount = a multiple of 100 and at least 100, checked on the ORIGINAL currency (USD here).
# In the app the ML service checks SAR, because that's what employees type.
# .astype(int) turns True / False into 1 / 0 for the model.
df["is_weekend"] = df["expense_date"].dt.dayofweek.isin([4, 5]).astype(int)
df["is_round_amount"] = ((df["amount_usd"] % 100 == 0) & (df["amount_usd"] >= 100)).astype(int)

print("Weekend %:", round(df["is_weekend"].mean() * 100, 1))
print(df.loc[df["is_weekend"] == 1, "expense_date"].dt.day_name().value_counts().to_dict())
print("Round:", df["is_round_amount"].sum(), "| %:", round(df["is_round_amount"].mean() * 100, 2))
print(df.loc[df["is_round_amount"] == 1, "amount_usd"].value_counts().head(5).to_dict())

Weekend %: 9.2
{'Friday': 12901, 'Saturday': 6390}
Round: 1560 | %: 0.75
{100.0: 256, 200.0: 175, 300.0: 166, 500.0: 134, 400.0: 102}


C:\Users\Zala\AppData\Local\Temp\ipykernel_32096\2342701406.py:5: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["is_weekend"] = df["expense_date"].dt.dayofweek.isin([4, 5]).astype(int)


In [15]:
# 5.13 All candidate features (11). Step 6 keeps 8 of them; the other 3 stay in the file so the
# Step 6 analysis can be re-run. Isolation Forest can't handle missing or infinite values,
# so both counts must be 0.
FEATURES = [
    "log_amount",
    "amount_vs_employee", "amount_vs_employee_category", "amount_vs_department",
    "category_share_for_employee",
    "same_category_day_count", "duplicate_count", "daily_request_count",
    "other_category_day_count",
    "is_weekend", "is_round_amount",
]
print("Missing:", int(df[FEATURES].isna().sum().sum()),
      "| Infinite:", int(np.isinf(df[FEATURES]).sum().sum()))
print(df[FEATURES].describe().T[["mean", "std", "min", "50%", "max"]].round(2))

Missing: 0 | Infinite: 0
                             mean   std    min   50%    max
log_amount                   5.94  1.52   0.04  5.87  15.22
amount_vs_employee          -0.00  1.11 -10.00 -0.02  10.00
amount_vs_employee_category -0.00  1.20 -10.00 -0.01  10.00
amount_vs_department         0.01  1.00  -4.27 -0.03   4.93
category_share_for_employee  0.57  0.32   0.00  0.59   1.00
same_category_day_count      2.76  4.70   1.00  1.00  80.00
duplicate_count              1.28  1.66   1.00  1.00  39.00
daily_request_count          3.10  4.92   1.00  2.00  80.00
other_category_day_count     0.34  1.16   0.00  0.00  57.00
is_weekend                   0.09  0.29   0.00  0.00   1.00
is_round_amount              0.01  0.09   0.00  0.00   1.00


In [16]:
# 5.14 Save the ids + baseline_level + the 10 features. The ids are for looking up flagged requests
# later (Step 8); they're not model inputs. The Part A helper columns (emp_n, ...) are not saved:
# the ML service keeps its own statistics (Step 9).
KEEP = ["expense_id", "requester_id", "department_id", "category", "expense_date",
        "amount_sar", "baseline_level"] + FEATURES

df[KEEP].to_csv("../data/processed/features.csv", index=False)
print("Saved:", df[KEEP].shape)

Saved: (209012, 18)
